<script src="{{site.baseurl}}/assets/js/code-runner-analytics.js"></script>

## Popcorn Hack 1: Read the API, Then Use It

**What I changed:** the jersey range is now `1-55` instead of `1-99`, and the court is `196` square feet instead of `144`.

**Prediction before running:** the jersey number will be a whole number from 1 to 55 (both ends included), and the court side will be `14.0` feet because 14 × 14 = 196. The random game and the rounded-up average (`25`) do not change.

In [ ]:
# CODE_RUNNER: Popcorn Hack 1 - Jersey range 1-55 and a 196 square foot court

import math
import random

# Read the API before calling a procedure
print("random.randint:", random.randint.__doc__)
print("math.ceil:", math.ceil.__doc__)
print()

points_per_game = [24, 18, 31, 27, 22]

# random.choice picks one item out of a list
print("Random game to review:", random.choice(points_per_game), "points")

# Changed: random.randint(1, 55) returns a whole number from 1 to 55, including both
jersey = random.randint(1, 55)
print("Random jersey number:", jersey)

# Changed: a 196 square foot court is 14 feet on each side
side = math.sqrt(196)
print("Square court side:", side, "feet")

# math.ceil rounds up to the next whole number
average = sum(points_per_game) / len(points_per_game)
rounded_up = math.ceil(average)
print("Points per game:", average, "-> rounded up:", rounded_up)

## Popcorn Hack 2: Write a Flask Route

I changed the roster to my own players and added a new route, `/coach`, above `app.test_client()`.

In [ ]:
# CODE_RUNNER: Popcorn Hack 2 - My roster and a new /coach route

from flask import Flask, jsonify

app = Flask(__name__)

# A route is a procedure with a web address attached to it
@app.route("/score")
def score():
    return jsonify({"home": 58, "away": 52, "quarter": 3})

@app.route("/roster")
def roster():
    return jsonify({"players": ["Aryan", "Jordan", "Priya", "Marcus"]})

# My new route
@app.route("/coach")
def coach():
    return jsonify({"coach": "Coach Rivera", "record": "6-2", "next_practice": "Tuesday 3:30 PM"})

# test_client visits the addresses without needing a browser
client = app.test_client()

print("/score says:", client.get("/score").get_json())
print("/roster says:", client.get("/roster").get_json())
print("/coach says:", client.get("/coach").get_json())

## Homework Hack: Soccer Season Stats Report

My version reports on a soccer season, so points became **goals**. It uses four standard library modules:

| Module | What I use it for |
| --- | --- |
| `statistics` | Average goals per game |
| `math` | Rounding goals per 90 minutes up to a whole number with `math.ceil` |
| `datetime` | Days left until my next game |
| `json` | Printing the report as JSON |

`summarize_season` has a docstring that describes its parameter and its return value, so someone can use it without reading the code.

In [ ]:
# CODE_RUNNER: Homework - Soccer season stats report

import json
import math
import statistics
from datetime import date

games = [
    {"date": "2026-09-05", "opponent": "Westview", "goals": 2, "minutes": 80},
    {"date": "2026-09-12", "opponent": "Rancho Bernardo", "goals": 1, "minutes": 75},
    {"date": "2026-09-19", "opponent": "Westview", "goals": 3, "minutes": 90},
    {"date": "2026-09-24", "opponent": "Del Norte", "goals": 0, "minutes": 60},
    {"date": "2026-09-26", "opponent": "Mt. Carmel", "goals": 2, "minutes": 85},
]

def summarize_season(season):
    """
    Summarize a soccer season.

    Parameters:
        season: list of dictionaries, each with "date" (YYYY-MM-DD text),
                "opponent" (text), "goals" (whole number), and "minutes" (whole number)
    Returns:
        dictionary with:
            total_goals: goals scored in every game added together
            average_goals: mean goals per game, rounded to 1 decimal place
            best_game: the most goals scored in a single game
            goals_by_opponent: dictionary of opponent name -> total goals against them
            goals_per_90_rounded_up: goals per 90 minutes played, rounded up
    """
    goals = [game["goals"] for game in season]
    minutes = [game["minutes"] for game in season]

    goals_by_opponent = {}
    for game in season:
        opponent = game["opponent"]
        goals_by_opponent[opponent] = goals_by_opponent.get(opponent, 0) + game["goals"]

    return {
        "total_goals": sum(goals),
        "average_goals": round(statistics.mean(goals), 1),
        "best_game": max(goals),
        "goals_by_opponent": goals_by_opponent,
        "goals_per_90_rounded_up": math.ceil(sum(goals) / sum(minutes) * 90),
    }

report = summarize_season(games)

# Subtracting two dates gives a timedelta, and .days is the whole number of days
next_game = date(2026, 10, 16)
report["next_game"] = "Torrey Pines on 2026-10-16"
report["days_until_next_game"] = (next_game - date.today()).days

print(json.dumps(report, indent=2))

# requirements.txt for a real version of this app:
# pandas for the season table and Flask for the scoreboard page.
# json, math, statistics, and datetime are NOT listed because they are part of
# Python's standard library. They come with Python, so pip never installs them.
requirements_txt = """pandas
Flask
"""
print()
print("requirements.txt:")
print(requirements_txt)